# SkillRadar — sådan er appen bygget
## Detaljeret gennemgang til mundtlig eksamen · Del 1: Webappen

**Kilde:** [Lucialululu/SkillRadar](https://github.com/Lucialululu/SkillRadar), `main`-branchen, læst 5. november 2026. Vigtigste filer: [`app.py`](https://github.com/Lucialululu/SkillRadar/blob/main/app.py), [`skillradar/ui.py`](https://github.com/Lucialululu/SkillRadar/blob/main/skillradar/ui.py), [`skillradar/cv_parser.py`](https://github.com/Lucialululu/SkillRadar/blob/main/skillradar/cv_parser.py), [`skillradar/matcher.py`](https://github.com/Lucialululu/SkillRadar/blob/main/skillradar/matcher.py), [`skillradar/demand.py`](https://github.com/Lucialululu/SkillRadar/blob/main/skillradar/demand.py), [`skillradar/report.py`](https://github.com/Lucialululu/SkillRadar/blob/main/skillradar/report.py), [`skillradar/robustness.py`](https://github.com/Lucialululu/SkillRadar/blob/main/skillradar/robustness.py).

**Formålet med notebooken:** At kunne fortælle *præcis hvad I har implementeret, hvorfor, hvordan data bevæger sig igennem appen, og hvordan resultaterne bliver regnet ud*. Alle tekstafsnit i notebooken er Markdown. Kodecellerne er selvstændige undervisningseksempler; de ændrer ikke jeres GitHub-repository og kræver ikke jeres private CV-data.

**Vigtig sondring:** Tekst markeret **Fra projektkoden** beskriver det observerede repository. **Didaktisk mini-eksempel** er en forenklet, kørbar demonstration af samme idé og må ikke forveksles med den fulde implementering eller faktiske resultater fra datasættet.

**Bemærkning om datagrundlaget:** Små eksempler med 4–8 fiktive jobopslag er *illustrationer*, ikke SkillRadars målte markedsresultater.

## 1. Det overordnede billede: Hvad gør appen?

Appen er en **Streamlit-webapp i Python**. En bruger kan uploade et CV, vælge en jobrolle og et sammenligningsdatasæt, gennemse og rette de automatisk genkendte softwareværktøjer og få en prioriteret oversigt over de efterspurgte værktøjer, der ikke står på CV'et.

**Den faktiske brugerrejse:**

1. **Upload** — PDF, DOCX, TXT eller Markdown læses og konverteres til tekst, udelukkende i hukommelsen under den normale app-session.
2. **Role** — brugeren vælger en af fem jobroller og enten alle erfaringsniveauer eller entry-level/internship-datasættet, hvis tilgængeligt.
3. **Check** — matcheren udtrækker konkrete, navngivne værktøjer; appen viser en tekstbid som dokumentation; brugeren kan fjerne forkerte fund og tilføje oversete værktøjer.
4. **Report** — appen beregner, hvor hyppigt værktøjerne forekommer i relevante jobopslag, deres statistiske intervaller, hvilke CV'et allerede nævner, hvilke der mangler, og et vægtet dækningsmål.

**Det appen ikke gør:** Den afgør ikke, hvad brugeren reelt mestrer; den vurderer indholdet af CV'et. Den forudsiger ikke sandsynligheden for at blive ansat. Den viser ikke pålidelige stigende/faldende trends over tid, fordi de to jobdatasæt primært er snapshots. Den almindelige `app.py` anvender ikke Gemini/LLM til at læse CV'et — LLM-varianterne findes andetsteds i projektets evaluering.

## 2. Arkitektur: Hvilke filer taler sammen?

| Fil | Ansvar | Hvorfor opdelt? |
|---|---|---|
| `app.py` | De fire skærmbilleder, knapper, widget-state og styring af rækkefølgen | Brugerflow adskilt fra beregninger |
| `skillradar/ui.py` | CSS, HTML/SVG, radarvisualisering, barer og kort | Design kan ændres/testes uden at røre dataalgoritmer |
| `skillradar/cv_parser.py` | PDF/DOCX/TXT/MD → renset tekst | Filformater håndteres ét sted |
| `skillradar/matcher.py` | Tekst → kanoniske O*NET-værktøjsnavne + evidens | Samme matchingmetode på CV'er og jobopslag |
| `skillradar/demand.py` | Offline build af jobopslagsdata, filtrering, frekvens og Wilson-intervaller | Webappen behøver ikke indlæse gigabyte rådata |
| `skillradar/report.py` | CV-værktøjer vs. værktøjsfrekvenser → have/missing/coverage | Forretningslogik kan testes uden web-UI |
| `skillradar/robustness.py` | Replikation, bootstrap-rangstabilitet og senioritetsforskel | Usikkerheds-/robusthedsanalyser isoleret |
| `main.py` | CLI til `build` og `report` | Samme beregninger kan bruges uden Streamlit |
| `app_testing.py` | Deltagerversion med ekstra feedbacktrin | Brugerundersøgelse holdes adskilt fra normal app |

**Dataflow:** `CV-fil → read_cv_bytes → clean_text → KeywordMatcher.extract_tools/evidence → menneskelig korrektion → CV-tool-set → demand_table(rolle, datasæt) → build_report → ui.radar/gap_row/coverage_meter → download .txt`.

**Hvorfor det er godt:** Det gør fejlfinding mere præcis. Hvis en PDF læses forkert, undersøger man parseren. Hvis Python ikke genkendes, undersøger man matcheren. Hvis coverage ser forkert ud, undersøger man rapportfunktionen.

**Alternative arkitekturer:** (A) én stor `app.py` er hurtigere at starte med, men sværere at teste/vedligeholde; (B) separat React-frontend + FastAPI-backend kan skalere til større systemer, men giver markant mere kompleksitet og deploymentarbejde. Den valgte Streamlit-løsning passer til en kursusprototype.

## 3. Hvordan starter appen?

Fra `app.py`'s dokumentation:

```bash
uv run streamlit run app.py
```

Appen bruger Python, Streamlit og pandas og importerer de interne moduler. Streamlit kører scriptet forfra, når brugeren interagerer med en widget (en *rerun*). Derfor er det ikke nok at bruge almindelige lokale variabler til at holde CV, valgt rolle og rettelser fra trin til trin; appen bruger `st.session_state`.

Inden brugerflowet starter, konfigurerer den siden med `st.set_page_config(page_title="SkillRadar", page_icon="📡", layout="wide")`, indlæser `ui.CSS` via `st.markdown(..., unsafe_allow_html=True)` og undersøger, om O*NET- og processed-datafilerne findes. Hvis de mangler, vises en fejl, og `st.stop()` standser resten af siden.

**Væsentligt:** Den app, brugeren ser, er derfor en *visning over en eksisterende beregningspipeline*; den indsamler ikke millioner af opslag i realtid.

## 4. Konfiguration: hvilke roller og datasæt kan brugeren vælge?

**Fra `app.py` (forkortet, faktiske værdier):**

```python
ONET = Path("data/onet/software_skills.csv")
TOP = 15
DATASETS = {
    "all": ("All experience levels", "LinkedIn, Jan 2024 · US, UK, Canada, Australia · largest sample", "primary", None),
    "entry": ("Entry level & internships", "LinkedIn, Apr 2024 · US only · closest to student roles", "secondary", ["entry", "internship"]),
}
```

**De fem roller** er `data scientist`, `data analyst`, `software engineer`, `ml engineer`, `ux designer`. Rollen identificeres i jobopslag via de titelbaserede mønstre i `demand.py` (`ROLE_PATTERNS`), ikke ved en fri semantisk forståelse af ethvert job.

- `"all"` peger på *primary* og ingen levels-filter (`None`): den større stikprøve.
- `"entry"` peger på *secondary* og levels `entry` samt `internship`: mere relevant for studerende, men et andet datasæt og geografisk område.
- `TOP = 15` er standardantallet af toprangerede værktøjer, der indgår i rapporten.

**Metodisk konsekvens:** Når brugeren skifter datasæt, ændrer de ikke kun senioritet — de skifter også kilde, tidspunkt og geografi. Derfor kan de to visninger ikke bruges som en ren årsagsmæssig sammenligning af junior vs. senior.

**Alternativ:** Brug ét datasæt med konsistente levels og samme geografi, hvis en direkte senioritetskontrast er vigtigere end stikprøvestørrelsen.

## 5. Streamlit cache — hvorfor læses data ikke om igen ved hvert klik?

**Fra `app.py`:**

```python
@st.cache_resource
def get_matcher() -> KeywordMatcher:
    return KeywordMatcher.from_csv(ONET)

@st.cache_resource
def get_postings(source: str):
    return load_posting_tools(source) if (PROCESSED / f"posting_tools_{source}.csv").exists() else None
```

`@st.cache_resource` gør, at en stor, genbrugelig ressource kan oprettes én gang og genbruges ved efterfølgende reruns. Det er relevant, fordi hver interaktion i Streamlit ellers kunne genindlæse matchervokabularet og datasættene. I appen cachelagres også replikations- og senioritetsanalyser samt bootstrap-rangstabilitet.

**Privatliv:** Cachefunktionerne her arbejder på fælles O*NET/jobdata — ikke på et uploadet CV. CV-tekst ligger i sessionens `st.session_state`, ikke i den delte resource-cache. 'Kun i hukommelsen' beskriver jeres egen håndtering i koden; det er ikke i sig selv en fuldstændig garanti for enhver hosting- eller logningskonfiguration.

**Alternativ:** `st.cache_data` bruges typisk til genberegnelige dataobjekter, mens `cache_resource` er til delte ressourcer. `cache_resource` kræver opmærksomhed på delte, muterbare objekter og samtidig adgang.

## 6. State machine: sådan husker appen trin og brugerens valg

**Fra `app.py`:**

```python
ss = st.session_state
for k, v in {"step": 1, "cv_text": None, "cv_name": None, "cv_key": None,
             "role": list(ROLE_PATTERNS)[0], "dataset": "all",
             "kept": {}, "added": []}.items():
    ss.setdefault(k, v)

def go(step: int) -> None:
    st.session_state.step = step
    st.rerun()

{1: step_upload, 2: step_role, 3: step_check, 4: step_report}[ss.step]()
```

`step` bestemmer hvilken af de fire funktioner, der tegner siden. `cv_text` indeholder udtrukket tekst, `cv_name` navnet, `cv_key` identitet for upload/eksempel, `role` og `dataset` valgene, `kept` ja/nej-værdier for hvert automatisk fund og `added` manuelt tilføjede værktøjer.

`go(3)` ændrer state til trin 3 og udløser en rerun. Ved ny upload nulstilles `kept` og `added` for at undgå, at rettelser fra det gamle CV påvirker det nye.

**Vigtigt detaljepunkt:** `step_role` bruger særskilte widget keys (`w_dataset`, `w_role`), der synkroniseres med de vedvarende værdier. Koden forklarer selv, at Streamlit kan glemme en widgets værdi, når widgetten midlertidigt ikke vises.

**Alternativ:** En `st.form`-baseret enkeltside kunne undgå flere trin, men ville give mindre guidet kontrol over fejlrettelser og mere information på én gang.

### Kørbart mini-eksempel: en forenklet state machine
Dette eksempel er **ikke Streamlit-koden**; det demonstrerer kun princippet bag state og nulstilling.

In [ ]:
state = {"step": 1, "cv_text": None, "kept": {}, "added": []}

def set_new_cv(text):
    state["cv_text"] = text
    state["kept"] = {}
    state["added"] = []
    state["step"] = 2

set_new_cv("Python and SQL")
state["kept"]["Python"] = True
print("Efter første CV:", state)
set_new_cv("Excel and R")
print("Efter nyt CV — gamle rettelser er fjernet:", state)

## 7. Trin 1 — Upload og CV-parsing

**Fra `app.py`:**

```python
up = st.file_uploader("Your CV", type=["pdf", "docx", "txt", "md"], label_visibility="collapsed")
if up is not None:
    try:
        set_cv(read_cv_bytes(up.getvalue(), up.name), up.name, f"upload:{up.name}:{up.size}")
    except ValueError as err:
        st.error(str(err))
```

En fil bliver til bytes med `up.getvalue()`, og parseren vælger læsemetode ud fra filendelsen. `read_cv_bytes()` returnerer renset tekst eller rejser en `ValueError`, hvis formatet ikke understøttes eller der findes færre end 50 tegn. Eksempelknappen anvender `EXAMPLE_CV` og går videre uden upload.

**Fra `cv_parser.py`:** `.pdf` → `pdfplumber` og `page.extract_text(x_tolerance=1.5)`; `.docx` → `python-docx`, inklusive tabelceller; `.txt/.md` → UTF-8-dekodning. Derefter udfører `clean_text()` bl.a. Unicode-normalisering (`NFKC`), ligaturrettelser, reparation af orddeling over linjeskift (`Kuber-\nnetes` → `Kubernetes`), fjernelse af bullets og sammenfoldning af mellemrum.

**Hvorfor:** CV'er kan have mange dokumentlayouts; matcheren har brug for ren tekst. Et dokument med skannede billeder uden tekstlag vil typisk ikke kunne udtrækkes korrekt med den valgte PDF-læser.

**Alternativer:** OCR til skannede PDF'er (mere dækning, men større kompleksitet og fejlrisiko); layout-aware dokumentparser (bedre kolonne-/tabelstruktur, dyrere); ren TXT-input (simpelt, men dårlig brugeroplevelse).

### Kørbart mini-eksempel: rens en CV-tekst
Dette gengiver centrale tekstregler i projektet, uden at kræve PDF/DOCX-pakker.

In [ ]:
import re, unicodedata

def demo_clean_text(text):
    text = unicodedata.normalize("NFKC", text)
    text = text.replace("\u00ad", "")
    text = re.sub(r"(\w)-\n(\w)", r"\1\2", text)
    text = re.sub(r"^[ \t]*[•●▪■◦‣∙·➢➤►▶✓✔★\-–—]+[ \t]*", "", text, flags=re.M)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

raw_cv = "• Python    and SQL\n• Kuber-\nnetes\n\n\n• Git"
print("FØR:", repr(raw_cv))
print("EFTER:", repr(demo_clean_text(raw_cv)))

## 8. Trin 2 — rolle- og datakildevalg

Brugeren vælger en titel fra de fem roller. Appen viser samtidig antallet af relevante opslag med `n_postings(role, dataset)`, der filtrerer den allerede processerede tabel. Derefter vælges en af de tilgængelige kilder via radioknapper.

**Vigtig kodeidé:**

```python
st.radio("Role", list(role_names), key="w_role", horizontal=True,
         captions=[f"{n_postings(r, ss.dataset):,} postings" for r in ROLE_PATTERNS])
...
st.radio("Compare against", list(ds_titles), key="w_dataset", horizontal=True,
         captions=[DATASETS[k][1] for k in datasets])
```

**Hvorfor vise antallet?** Det hjælper med at kommunikere evidensgrundlag: en rolle med 50 opslag giver en mere usikker frekvens end én med 5.000. Brugerens valg styrer *hvilke opslag, der bliver sammenlignet med*, og derfor hele rapporten.

**Alternativer:** Et frit tekstfelt til roller ville være fleksibelt, men kræver jobtitelnormalisering og kontrol for ukendte roller. Et dropdown med hundredvis af roller ville give bredere dækning, men mindre robust evaluering inden for kursusprojektets scope.

## 9. Trin 3 — hvad er en 'skill' i appen?

Den nuværende webapp fokuserer på **navngivne softwareværktøjer og teknologier**, fx Python, SQL, Tableau, Docker og Power BI. Den tæller ikke alle brede kompetencer: ord som *teamwork* eller *machine learning* er ikke nødvendigvis softwareværktøjer og vises derfor ikke som selvstændige værktøjer i appens valideringsliste.

Vokabularet kommer fra `data/onet/software_skills.csv`. `KeywordMatcher` anvender de værktøjer, som O*NET markerer som *Hot Technology* eller *In Demand*, og danner varianter/aliasnavne til tekstmatch. **O*NET er den fælles navnestandard; keyword-matching er metoden til at finde nævnelserne.**

**Hvorfor normalisering?** I et CV står måske “AWS”, mens O*NET bruger en længere kanonisk betegnelse. Begge bør tælles som samme værktøj, ellers undervurderes overlap mellem CV og jobopslag.

**Begrænsning:** Et nøgleordsfund viser omtale, ikke færdighedsniveau. Et CV, der siger “I have not used Docker”, kan stadig indeholde ordet Docker. Dette er en vigtig grund til brugerens kontroltrin.

**Alternativer:** (A) rå exact matching — hurtig, men overser aliaser; (B) embeddings/semantiske modeller — kan finde parafraser, men kan forbinde beslægtede snarere end identiske værktøjer; (C) LLM-kontekstforståelse — kan håndtere negation/kontekst, men er langsommere, dyrere og sværere at reproducere. Projektet tester også LLM-/hybridvarianter i separate evalueringer; webappens standard bruger `KeywordMatcher`.

## 10. Hvordan virker `KeywordMatcher` helt konkret?

**Faktisk kodeprincip fra `matcher.py`:**

```python
for n in range(1, self.max_n + 1):
    for i in range(len(toks) - n + 1):
        for g in (" ".join(low[i:i + n]), " ".join(toks[i:i + n])):
            if g in self.alias_to_tool:
                found[g] = self.alias_to_tool[g]
```

1. `tokens(text)` opdeler tekst i tokens ved hjælp af et specialiseret regex (bevarer bl.a. symboler relevante for `C++`, `C#`).
2. Der genereres *n-grams* (sammenhængende sekvenser af 1 til højst `MAX_NGRAM = 8` ord).
3. For hver sekvens testes både en version med små bogstaver og den oprindelige casing mod `alias_to_tool`.
4. Hvis et alias genkendes, gemmes dets kanoniske O*NET-værktøjsnavn.
5. `extract_tools(text)` returnerer `set(self.explain(text).values())`, så samme værktøj kun findes én gang pr. CV/opslag.
6. `evidence(text)` returnerer et uddrag omkring det første match, med markerede ord, så brugeren kan se, *hvorfor* appen mener, at værktøjet står på CV'et.

**Konflikter i aliaser:** Når to værktøjer deler en betegnelse, gives det mest udbredte værktøj forrang ifølge vokabularets sortering. Visse flertydige aliaser fjernes manuelt (`MANUAL_DROP`); der laves ikke automatisk frasortering af alle hyppige aliaser. Case-sensitive regler hjælper eksempelvis med korte ord som `R` og `Go`.

**Trade-off:** Reglerne er lette at inspicere og teste, men kræver vedligeholdelse; kontekstafhængige falske positiver kan stadig forekomme.

### Kørbart mini-eksempel: aliaser og dobbelttælling
Følgende lille map er **fiktivt og forsimplet**. Den rigtige matcher bygger aliaserne automatisk fra O*NET og anvender flere specialregler.

In [ ]:
aliases_demo = {"python": "Python", "sql": "SQL", "aws": "AWS", "amazon web services": "AWS", "docker": "Docker"}
cv = "I used Python, AWS (Amazon Web Services), Python again and SQL."
found = {tool for alias, tool in aliases_demo.items() if alias in cv.lower()}
print("Fundne kanoniske værktøjer:", sorted(found))
print("Bemærk: AWS og Amazon Web Services bliver kun ét værktøj i et set.")

## 11. Human in the loop — hvorfor brugeren selv kontrollerer listen

**Fra `app.py`:**

```python
found = matcher.extract_tools(ss.cv_text)
evidence = matcher.evidence(ss.cv_text)
...
ss.kept[t] = c1.checkbox(display_name(t), key=k, label_visibility="collapsed")
...
ss.added = st.multiselect("Add a tool we missed", options, key=k)
```

`found` er maskinens rå værktøjsliste. Appen viser et checkbox-felt og et evidensuddrag for hvert fund. `ss.kept` gemmer brugerens beslutning om at beholde/fjerne hvert fund. `ss.added` gemmer værktøjer, som brugeren vælger fra det eksisterende katalog, men som matcheren overså.

**To fejlkategorier:** *False positive* betyder, at appen fandt et værktøj, som CV'et ikke dokumenterer i den relevante betydning (fx 'Go' som almindeligt ord). *False negative* betyder, at appen ikke fandt et værktøj, som faktisk fremgår af CV'et (fx utraditionel stavning).

**Hvorfor dette designvalg?** I prioriterer en forklarlig, brugerrettet proces frem for at skjule usikre automatiske match. I et projekt om personlige anbefalinger påvirker hver fejl direkte, hvilke læringsbehov brugeren får vist.

**Vigtig konsekvens:** Brugerens tilføjelser begrænses til matcherens katalog. Vedkommende kan ikke frit tilføje alle tænkelige nye teknologier, som endnu ikke findes i O*NET-vokabularet.

**Alternativ:** Automatisk rapport uden kontrol er hurtigere men risikerer misvisende anbefalinger; fri teksttilføjelse er fleksibel men kræver selv normalisering og validering.

### Kørbart mini-eksempel: hvad ændrer en manuel rettelse?
Dette er samme mængdelogik som i `step_report()`; værdierne er illustrative.

In [ ]:
found = {"Python", "SQL", "Go"}
kept = {"Python": True, "SQL": True, "Go": False}  # 'Go' var et fejlagtigt fund
added = {"Docker"}                                     # Docker blev overset
cv_tools = {t for t in found if kept.get(t, True)} | added
print("Automatisk fund:", sorted(found))
print("Endelig godkendt CV-liste:", sorted(cv_tools))
assert cv_tools == {"Python", "SQL", "Docker"}

## 12. Hvad er 'jobdata' i appen? Offline build vs. online query

Jeres `demand.py` har to forskellige opgaver:

**BUILD, offline:** `python main.py build --source primary` og tilsvarende `secondary`. De store rå jobopslag læses, relevante jobroller udvælges, værktøjer udtrækkes fra opslagsteksten med samme matcher som til CV'et, og der gemmes en mindre tabel i `data/processed/posting_tools_<source>.csv`. Rækkerne indeholder bl.a. opslagets ID, kilde, rolle, level, land, titel og de fundne værktøjer adskilt med semikolon. Gentagne identiske opslag deduplikeres via en tekst-hash ved indlæsning.

**QUERY, i webappen:** `load_posting_tools(source)` henter den mindre CSV. `filter_postings()` filtrerer på rolle og eventuelt level/land. `demand_table()` beregner hyppigheden af hver teknologi på de filtrerede opslag.

**Hvorfor vigtigt:** Den primære raw jobtekstfil er meget stor (README beskriver ca. 5 GB). Det ville være urealistisk at genlæse og matche alt for hvert knaptryk. I betaler beregningsomkostningen under build og bruger derefter et hurtigt, allerede klargjort datasæt.

**Kritisk ved ændringer:** Når reglerne i matcheren ændres, skal jobopslagene behandles på ny (`build`), ellers sammenligner I en opdateret CV-matcher med gamle posteringsudtræk.

**Alternativ:** Database med indekser kan være mere skalerbar, men en processed CSV er enkel, transparent og tilstrækkelig til prototypen.

## 13. Hvordan måler I efterspørgsel?

**Fra `demand.py`:** `demand_from_sets()` samler et `set` af værktøjer for hvert jobopslag, tæller hvor mange opslag der nævner hvert værktøj (`n_with_tool`), og beregner `share = k/n`. Et opslag tæller højst én gang pr. værktøj.

**Definition:**

\[
\operatorname{Demand}(t\mid r)=\frac{\#\text{ relevante opslag for rolle }r\text{ der nævner værktøj }t}{\#\text{ relevante opslag for rolle }r}
\]

Hvis 6 ud af 8 relevante opslag nævner Python, er dens demand **75 %**. Det betyder ikke, at 75 % af alle jobs efterspørger Python; tallet gælder det filtrerede datasæt.

**Hvorfor:** Målet kan forklares, verificeres og sammenlignes. Det tæller ikke ordforekomster, fordi lange opslag ellers kan dominere.

**Alternativer:** TF-IDF, virksomheds-/opslagsvægtning eller modelbaserede relevansscores; hver kræver flere antagelser.

### Kørbart mini-eksempel: efterspørgsel fra fiktive jobopslag

In [ ]:
from collections import Counter
postings = [
    {"Python", "SQL", "Docker"},
    {"Python", "SQL"},
    {"Python", "AWS"},
    {"Python", "Docker"},
    {"SQL", "AWS"},
    {"Python", "SQL"},
    {"Python", "SQL", "Docker"},
    {"SQL"},
]
n = len(postings)
counts = Counter(t for tools in postings for t in tools)
shares = {t: k/n for t, k in counts.items()}
for tool, share in sorted(shares.items(), key=lambda x: -x[1]):
    print(f"{tool:8} {counts[tool]}/{n} = {share:.1%}")

## 14. Hvorfor viser I 95 %-intervaller?

Ud over en procent beregner `demand.py` et **Wilson score interval** med `z=1.96`, normalt fortolket som et omtrentligt 95 %-konfidensinterval for en andel. Det er ofte bedre end det simple Wald-interval, især når andelen ligger nær 0/1 eller datasættet er lille.

Den konkrete kode er:

```python
p = k / n
denom = 1 + z**2 / n
centre = (p + z**2 / (2 * n)) / denom
half = z * math.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / denom
return (max(0.0, centre - half), min(1.0, centre + half))
```

Appens `gap_row` viser både procent og interval som en bjælke med en markør. Hvis der er færre end `SMALL_SAMPLE = 100` relevante opslag, advarer rapporten om, at procenttallene er grove.

**Meget vigtig eksamensnuance:** Et 95 %-konfidensinterval betyder ikke strengt, at der er 95 % sandsynlighed for, at den sande parameter ligger i netop dette beregnede interval. Det handler om metoden over gentagne stikprøver, og forudsætter en passende stikprøvemodel. Jobopslag i en scrape er ikke nødvendigvis uafhængigt og tilfældigt udvalgt fra hele arbejdsmarkedet. Intervallerne beskriver derfor *sampling-usikkerhed under modellen*, ikke al skævhed fra scraping, dubletter, rollefiltrering og ekstraktionsfejl.

### Kørbart mini-eksempel: Wilson interval ved forskellig stikprøvestørrelse

In [ ]:
import math

def wilson_demo(k, n, z=1.96):
    if n == 0: return (0.0, 0.0)
    p = k/n
    d = 1 + z*z/n
    center = (p + z*z/(2*n))/d
    half = z*math.sqrt(p*(1-p)/n + z*z/(4*n*n))/d
    return max(0.0, center-half), min(1.0, center+half)

for k, n in [(6,8), (75,100), (750,1000)]:
    lo, hi = wilson_demo(k,n)
    print(f"{k:4}/{n:4}: estimat={k/n:.1%}, 95%-Wilson={lo:.1%}–{hi:.1%}")

## 15. Hvordan laver I den personlige skill-gap rapport?

**Fra `report.py`, kerneprincip:**

```python
ranked = demand[demand["share"] >= min_share].head(top).copy()
ranked["on_cv"] = ranked["tool"].isin(cv_tools)
total = ranked["share"].sum()
coverage = float(ranked.loc[ranked["on_cv"], "share"].sum() / total) if total else 0.0
```

1. Sorterede demand-data indeholder de mest efterspurgte værktøjer for rollen.
2. Værktøjer under `min_share=0.05` filtreres fra.
3. Højst `top=15` værktøjer udvælges.
4. `on_cv` markerer de værktøjer, den rettede CV-liste indeholder.
5. `have` er de valgte topværktøjer, der findes på CV'et; `missing` er resten, rangeret efter demand.
6. `other_cv_tools` indeholder CV-værktøjer uden for denne topgruppe, med deres observerede frekvens (evt. 0).

**Bemærk:** 'Missing' betyder 'ikke dokumenteret i CV'et' — ikke at personen aldrig har brugt det. Appen bruger *det manuelt validerede CV-værktøjssæt* som input til rapporten.

## 16. Hvordan beregnes coverage-score præcist?

Coverage er en **efterspørgselsvægtet dækning af topværktøjerne**, ikke antallet af værktøjer delt med 15.

\[
\mathrm{Coverage}=\frac{\sum_{t\in \mathrm{TopTools}\cap\mathrm{CV}}\mathrm{Demand}(t)}{\sum_{t\in\mathrm{TopTools}}\mathrm{Demand}(t)}.
\]

**Eksempel:** Python forekommer i 75 %, SQL i 75 %, Docker i 37,5 % og AWS i 25 %. Hvis CV'et nævner Python og SQL, men ikke Docker og AWS, bliver coverage \((0.75+0.75)/(0.75+0.75+0.375+0.25) = 70.6\%\). **Det betyder ikke 70,6 % sandsynlighed for ansættelse** eller at man matcher 70,6 % af hvert jobopslag. Det er et defineret, aggregeret scoremål.

**Hvorfor vægte med demand?** At mangle et værktøj, som 80 % af opslagene nævner, bør have større indflydelse end at mangle ét, som 10 % nævner.

**Begrænsninger:** Skills bliver behandlet som binære (nævnt/ikke nævnt), stillingskrav kan være 'nice to have', og to meget korrelerede tools kan give dobbelt vægt. Alternatives: uvejet andel, IDF-vægtning, ekspertvægte, per-opslag matchscore eller modeller med must-have/optional — men alle har deres egne antagelser.

### Kørbart mini-eksempel: hele report-logikken på vores fiktive opslag

In [ ]:
cv_tools = {"Python", "SQL"}
ranked = sorted(shares.items(), key=lambda x: -x[1])[:4]
total_weight = sum(share for _, share in ranked)
have = [(t,s) for t,s in ranked if t in cv_tools]
missing = [(t,s) for t,s in ranked if t not in cv_tools]
coverage = sum(s for _,s in have)/total_weight
print("Allerede på CV'et:", have)
print("Mangler:", missing)
print(f"Demand-vægtet coverage: {coverage:.1%}")
print("Dette er kun en illustrationsberegning fra de 8 fiktive opslag.")

## 17. Trin 4 — Hvordan præsenteres resultaterne?

`step_report()` i `app.py`:

- bygger den endelige CV-tool-mængde som `cv_tools = {t for t in found if ss.kept.get(t, True)} | added`;
- kalder `demand_table(...)`, derefter `build_report(..., top=TOP, min_share=0.05)`;
- samler `have` og `missing` til én rangordnet liste;
- viser en radar (`ui.radar`) for topværktøjer, en **coverage-score** og `ui.coverage_meter`;
- viser “Already on your CV” som chips og “Missing, most requested first” med frekvensbjælker og usikkerhed;
- tilføjer en senioritetslabel kun, hvis analysen markerer forskellen som signifikant;
- viser en note om datakildernes geografiske begrænsninger;
- tilbyder tekst-download med `st.download_button("Download", format_report(rep), ...)`.

**Detalje fra `ui.py`:** Farverne er bevidst konsistente: **teal `#18D9B5`** = findes på CV, **amber `#FFB13D`** = mangler. En coverage-meter tegner segmenter med bredde proportional med demand; summen af teal segmenters bredder er derfor lig coverage-målet.

**Hvorfor denne visualisering?** En studerende kan hurtigt se både status (*har/mangler*) og prioritet (*hvor tit bliver værktøjet nævnt?*), mens procent og interval giver mere præcise oplysninger til den kritiske bruger.

## 18. Hvad er radaren — og hvad betyder afstand til centrum?

Radaren er primært en **visuel repræsentation** af rangering og status, ikke en separat prediktiv model. I `skillradar/ui.py` laves HTML/SVG. Ifølge modulbeskrivelsen placeres hyppigere efterspurgte værktøjer tættere på centrum, og farven viser, om værktøjet findes på CV'et. Ringene repræsenterer efterspørgselsniveauer (75/50/25 %). `ui.radar(...)` modtager tuples `(display_name, share, on_cv)`.

**Vigtigt i eksamen:** `radar` er altså en præsentationskomponent oven på de samme tal, som bruges i tabellen og coverage-beregningen. Den må ikke beskrives som en selvstændig ML-model.

**Alternativ visualisering:** Et sorteret horisontalt søjlediagram er ofte lettere at sammenligne præcise procenter i. Radaren er til gengæld mere karakteristisk og kan gøre prototypen engagerende, mens den detaljerede liste holder de nøjagtige tal synlige.

## 19. Hvorfor viser appen 'How sure are we?'

Appen kalder tre analyser fra `robustness.py`:

- `rank_intervals(..., n_boot=300)`: bootstrap-baseret usikkerhed for placeringer i top-15. `get_stable_ranks` tæller, hvor mange af de øverste rangplaceringer der i rækkefølge har rank-interval med bredde højst 1 (`hi - lo <= 1`).
- `replication(..., n_boot=300)`: sammenligner rangordener mellem to uafhængigt indsamlede datasæt, bl.a. med Spearman-rangkorrelation `ρ`. Appen viser værdien, hvis der er data.
- `seniority(...)`: sammenligner efterspørgsel mellem entry- og seniorroller og viser kun en særligt formuleret tag, når forskellen er markeret signifikant. Appen afviser også senioritetsresultatet, hvis enten junior- eller seniorgruppen har under 30 opslag.

**Hvorfor:** Et tal kan virke meget præcist, selvom rangordenen ikke er stabil. Rangbootstrap og replikation giver derfor supplerende information om robusthed.

**Nuance:** En høj korrelation viser lignende rangorden på tværs af de to scrapes; den beviser ikke, at datasættene er repræsentative for hele markedet. Man bør heller ikke sammenblande “rank-stabilitet” med det enkelte tools Wilson-interval.

## 20. `ui.py`, HTML, CSS og sikker visning af CV-tekst

Projektet bruger **Streamlit som applikationsramme**, men i stedet for kun standardwidgets bygges det visuelle udtryk i `ui.py` med CSS, HTML og SVG. Det er sådan I får en specialdesignet radar, step-bar, kort, chips og statusfarver uden en separat JavaScript-frontend.

**Fra `ui.py`:** `evidence_html()` kalder først `e(snippet)` (HTML-escape) og erstatter derefter de sikre `**...**`-markører fra matcheren med `<mark>...</mark>`. På samme måde escaper `tool_row`, `card`, `chips` mv. strenge før HTML-visning.

**Hvorfor?** CV-tekst er brugerleveret input. Hvis man sætter den direkte ind i `unsafe_allow_html=True`, kan specialtegn og HTML-tags potentielt blive fortolket som markup. HTML-escaping sørger for, at fx `<script>` vises som tekst. Det er en vigtig adskillelse mellem *designet HTML*, som I selv kontrollerer, og *brugerdata*, som skal behandles som tekst.

**Alternativer:** Kun native Streamlit-komponenter er enklere og reducerer behovet for HTML-escaping, men giver mindre visuel kontrol; en rigtig frontend med React giver mere designfrihed, men kræver ekstra applikationslag.

### Kørbart mini-eksempel: HTML-escaping
Dette demonstrerer det sikkerhedsprincip, `ui.py` anvender.

In [ ]:
import html
cv_snippet = 'Used <script>alert("x")</script> alongside **Python**'
escaped = html.escape(cv_snippet)
print("Uden escaping:", cv_snippet)
print("Med escaping:", escaped)
print("Den escapede version kan vises som tekst i HTML uden at script-tagget tolkes som HTML.")

## 21. Privatliv: hvad bliver gemt?

Den almindelige `app.py` læser uploadet CV til tekst i sessionens hukommelse og tilbyder en downloadfil genereret fra rapporten. Det ses ikke i `app.py`, at CV-tekst skrives til en persistent fil eller logs. Desuden nulstilles CV-information og manuelle tilføjelser, når brugeren trykker “Start over”.

**Separat brugertest:** `app_testing.py` udvider flowet til **fem trin** (`Upload → Role → Check → Report → Feedback`). Dets modulbeskrivelse siger, at studie-feedback gemmes ved samtykke, men ikke CV-tekst. README angiver, at testrespons kan gemmes i Google Sheets på hosted deployment eller lokal CSV. Det er derfor **en anden appvariant** og bør ikke beskrives som om almindelig `app.py` automatisk sender CV-data til Google Sheets.

**Nuance:** `st.session_state` er sessionspecifik hukommelse i applikationen, ikke det samme som lokal browser-only opbevaring. Udsagnet om, at CV aldrig lagres, er et udsagn om projektets implementering; drift, logs og tredjepartskomponenter kræver selvstændig vurdering, hvis det skal bruges som en formel databeskyttelsesgaranti.

## 22. Begrænsninger, som I bør nævne til eksamen

1. **Detektionsfejl:** Regler kan finde forkert teknologi i en tvetydig sætning eller overse en ukendt betegnelse. Human-in-the-loop er valgt som afbødning.
2. **Taxonomi-dækning:** O*NET's liste er en afgrænset teknologivokabular og er amerikansk; ikke alle danske eller helt nye værktøjer dækkes.
3. **Binarisering:** 'SQL nævnt' kan dække alt fra begynder til ekspert. Appen vurderer ikke kompetenceniveau.
4. **Datarepræsentativitet:** 2024-LinkedIn-scrapes har geografisk og indsamlingsmæssig bias. Det er ikke den aktuelle efterspørgsel i Danmark.
5. **Snapshot frem for tidsserie:** I har ikke grundlag for at udlede egentlige månedlige trends fra de to forskellige scrapes; jeres nuværende app fokuserer i stedet på efterspørgselsrangering og robusthed.
6. **Statistisk usikkerhed:** Wilson-intervaller indfanger ikke alle scraping- og modelbias; replikation er en supplerende, ikke endelig, kontrol.
7. **CV ≠ personens fulde viden:** En manglende teknologi kan være kendt, blot ikke nævnt på CV'et.
8. **Sammenligning på tværs af kilder:** 'All' vs. 'Entry' ændrer både source/land/tidspunkt og erfarenhedsfilter, ikke kun senioritet.

**Stærkt forsvar:** I har gjort begrænsninger synlige i selve brugeroplevelsen (kontroltrin, evidens, interval, datanote) og adskilt målemetode fra præsentation.

## 23. En sammenhængende gennemgang, I kan sige højt (ca. 2–3 minutter)

> Vi har bygget en webprototype i Streamlit. Appen er kun frontend og orkestrering — selve databehandlingen ligger i separate Pythonmoduler. Først har vi en offline pipeline, som læser jobopslag og matcher omtaler af softwareværktøjer til en fælles O*NET-vokabular. Vi gemmer et lille behandlet datasæt, så webappen ikke skal gennemgå gigabyte jobtekst, hver gang brugeren klikker.
>
> Brugeren uploader derefter et CV som PDF, Word eller tekst. Vi udtrækker teksten og renser almindelige dokumentartefakter. Vores keyword-matcher søger efter værktøjsnavne og aliaser som Python, SQL og AWS. Den samme matcher er brugt til jobopslagene, så betegnelserne er konsistente.
>
> Vi ved, at ordgenkendelse kan fejle, så før rapporten ser brugeren en liste med de fundne værktøjer og konkrete tekstuddrag fra CV'et. Brugeren kan fjerne falske positiver og tilføje oversete værktøjer. Det endelige, korrigerede sæt bliver sammenlignet med jobopslag for den valgte rolle.
>
> For hvert værktøj beregner vi andelen af relevante opslag, som nævner det. Vi viser også et 95 %-Wilson-interval. Topværktøjerne deles i dem, CV'et nævner, og dem, CV'et ikke nævner. Vi beregner en efterspørgselsvægtet coverage-score: summen af efterspørgslen for de topværktøjer, der findes på CV'et, divideret med summen for alle topværktøjerne. Scoren er ikke en ansættelsessandsynlighed.
>
> Rapporten bruger en radar og farvekodede lister til at gøre prioriteringen forståelig. Vi supplerer med bootstrap-rangstabilitet og replikation mellem datasæt, men er åbne om, at scrapingdata og O*NET-vokabularet har begrænsninger, især for det danske arbejdsmarked. Målet er beslutningsstøtte til, hvad man skal undersøge eller lære, ikke en garanti for at få jobbet.

## 24. Mundtlige eksamensspørgsmål — med korte, præcise svar

**Hvorfor Streamlit i stedet for React?** For at få en brugbar Pythonbaseret prototype med få ekstra teknologier, så arbejdet kunne prioriteres mod datakvalitet, matching og evaluering.

**Hvad er en taxonomy?** Et fælles katalog over kanoniske værktøjsnavne (her O*NET-softwareværktøjer), som gør CV og opslag sammenlignelige. Taxonomien definerer *hvad* man kan genkende; keyword-matcherens regler definerer *hvordan* det genkendes.

**Er det AI?** Standardwebappen udtrækker værktøjer med regelbaseret keyword-matching og aliasnormalisering. Projektet undersøger også LLM-baserede alternativer særskilt, men I bør ikke påstå, at webappens CV-upload i sig selv bruger Gemini.

**Hvorfor et `set`?** Fordi et værktøj kun skal tælle én gang pr. CV og jobopslag; gentagne omtaler ville ellers forvrænge andelen.

**Hvad betyder 42 % demand?** 42 % af de filtrerede opslag i det valgte datasæt nævner værktøjet, ikke nødvendigvis 42 % af alle job i markedet.

**Hvordan adskiller demand og coverage sig?** Demand er ét værktøjs forekomst blandt opslag. Coverage sammenfatter efterspørgslen efter de topværktøjer, som CV'et allerede dokumenterer, relativt til summen for alle topværktøjerne.

**Hvorfor ikke bare score 12/15?** Det ville vægte en sjælden teknologi lige så højt som en meget udbredt. Demand-vægtning matcher bedre målet om at prioritere læring.

**Hvad gør I ved falske positiver?** Case-regler, manuelt fjernede aliaser, visning af evidens og brugerens mulighed for at fjerne fund i Check-trinnet.

**Hvad er den største metodebegrænsning?** Den observerede procent er afhængig af kvaliteten af jobdatasæt, rollefiltrering og værktøjsekstraktion. Ingen score kan blive mere troværdig end disse input.

**Hvorfor viser I confidence intervals og bootstrap?** De beskriver forskellige typer usikkerhed: Wilson-intervaller omkring observerede andele og bootstrap omkring rangordenens stabilitet.

**Hvordan ville I videreudvikle appen?** Bedre kontekstafhængig ekstraktion med dokumenteret evalueringsforbedring, en mere EU-relevant taxonomi eller dataindsamling og tydeligere skelnen mellem nødvendige og ønskelige skills. Det er forslag, ikke implementerede funktioner.

## 25. Tjekliste til eksamen: Vis appen mens I forklarer

- [ ] **Upload:** demonstrér `Example CV`, og forklar, hvordan `read_cv_bytes` og `clean_text` fungerer.
- [ ] **Role:** skift rolle og forklar, hvorfor antallet af relevante opslag ændrer sig.
- [ ] **Check:** find et værktøj og peg på evidensuddraget; fjern et fund eller tilføj et manglende.
- [ ] **Report:** peg på, hvordan samme værktøj skifter mellem `have` og `missing`, når CV-sættet ændres.
- [ ] **Tal:** forklar forskellen mellem en demand-procent og den samlede coverage-score.
- [ ] **Usikkerhed:** forklar, hvad intervallet, “How sure are we?” og den begrænsede datadækning betyder.
- [ ] **Design:** fremhæv, at UI, parser, matcher, demand og report er adskilte moduler.
- [ ] **Afgrænsning:** understreg, at den nuværende webapp ikke laver en robust tidsserieprognose eller bestemmer ansættelsessandsynlighed.

**Tip:** Øv jer i at forklare ét konkret værktøj hele vejen fra en sætning i CV'et til dets rang i rapporten. Det tester forståelsen af hele pipelinen.

## 26. Projektets kode og kilder — åbn de konkrete funktioner

- [`app.py`](https://github.com/Lucialululu/SkillRadar/blob/main/app.py): `get_matcher`, `get_postings`, `go`, `set_cv`, `step_upload`, `step_role`, `step_check`, `step_report`.
- [`skillradar/cv_parser.py`](https://github.com/Lucialululu/SkillRadar/blob/main/skillradar/cv_parser.py): `read_cv_bytes`, `_read_pdf`, `_read_docx`, `clean_text`.
- [`skillradar/matcher.py`](https://github.com/Lucialululu/SkillRadar/blob/main/skillradar/matcher.py): `build_vocab`, `aliases`, `KeywordMatcher.explain`, `.extract_tools`, `.evidence`.
- [`skillradar/demand.py`](https://github.com/Lucialululu/SkillRadar/blob/main/skillradar/demand.py): `build`, `load_posting_tools`, `filter_postings`, `wilson`, `demand_from_sets`, `demand_table`.
- [`skillradar/report.py`](https://github.com/Lucialululu/SkillRadar/blob/main/skillradar/report.py): `build_report`, `format_report`, `SkillGapReport`.
- [`skillradar/ui.py`](https://github.com/Lucialululu/SkillRadar/blob/main/skillradar/ui.py): `radar`, `coverage_meter`, `gap_row`, `evidence_html`.
- [`skillradar/robustness.py`](https://github.com/Lucialululu/SkillRadar/blob/main/skillradar/robustness.py): `rank_intervals`, `replication`, `seniority`.
- [`app_testing.py`](https://github.com/Lucialululu/SkillRadar/blob/main/app_testing.py): brugerundersøgelsesversionen med feedbacktrin.
- [`README.md`](https://github.com/Lucialululu/SkillRadar/blob/main/README.md): build/run/afgrænsninger/eksperimenter.

**Afgrænsning af denne notebook:** Den forklarer webappens konstruktion grundigt; den er ikke en komplet celle-for-celle gennemgang af alle forsknings-/evalueringsscripts i repositoryet. Det naturlige næste kapitel er *datapipeline og matcher-evaluering*, hvor I kan gå dybere ned i aliasgenerering, labels, precision/recall og LLM-ablation.